# gravsim: what the simulator generates

`gravsim` (folder `sim/` on branch `feat/sensor-simulator`) stands in for the instruments we don't have. This notebook shows what it produces and checks it against the bed-rest data it was fitted on.

Everything below is **synthetic**. The curves come from NASA bed-rest studies (head-down tilt, 21 to 90 days), not from spaceflight, and anything past 90 days without gravity is the fitted curve continued.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))          # run from sim/examples
import numpy as np
import pandas as pd
import altair as alt
import gravsim as gs

alt.renderers.enable("mimetype")
pd.set_option("display.precision", 1)
BLUE, ORANGE, AQUA, VIOLET, GRAY, INK, SURFACE = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#898781", "#52514e", "#ffffff"
print(f"gravsim {gs.__version__}, fitted {gs.PARAMS['fitted_at']} from {gs.PARAMS['data']}")

gravsim 0.1.0, fitted 2026-11-14T19:21:27Z from data/nlsp (NASA Life Sciences Portal: MR079G, MR035G, MR080G), head-down bed rest


## 1. Four crew members, one mission

A 180-day ISS mission with four people whose statuses differ in the ways the generator supports:

- **Ana** follows the exercise routine (normal), as the median person of the exercise group.
- **Ben** does no exercise (not normal), as the median person of the bed-rest control groups.
- **Caro** follows the routine, drops to 20% between flight days 60 and 120, and sprains an ankle on day 95.
- **Dani** follows the routine but is a fast responder, losing faster than 9 in 10 subjects.

In [2]:
mission = gs.Mission("iss_180", launch_date="2027-03-01")
ana = mission.add_crew("Ana", sex="F", adherence=1.0, responder=0, seed=1)    # responder=0: exactly the group median
ben = mission.add_crew("Ben", sex="M", adherence=0.0, responder=0, seed=2)
caro = mission.add_crew("Caro", sex="F", adherence=[(0, 1.0), (60, 0.2), (120, 1.0)], seed=3)
caro.add_event(day=95, measures="legs", severity_pct=12, recovery_days=20, label="ankle sprain")
dani = mission.add_crew("Dani", sex="M", adherence=1.0, responder="fast", seed=4)
CREW = alt.Scale(domain=["Ana", "Ben", "Caro", "Dani"], range=[AQUA, ORANGE, BLUE, VIOLET])
mission.crew

[CrewMember('Ana', sex=F, age=35, responder=0),
 CrewMember('Ben', sex=M, age=34, responder=0),
 CrewMember('Caro', sex=F, age=47, responder='typical'),
 CrewMember('Dani', sex=M, age=28, responder='fast')]

The true curves (no measurement noise) for knee extension strength. The black dots are the bed-rest cohorts the curve was fitted on, at the length of their bed rest.

In [3]:
days = np.arange(-30, 366, 1.0)
true = pd.DataFrame([{"crew": c.id, "day": d, "change_pct": c.change("knee_extension_60", d)}
                     for c in mission.crew for d in days])
obs = pd.DataFrame(gs.MEASURES["knee_extension_60"]["observed"]).dropna(subset=["days"])
landing = pd.DataFrame({"day": [0, mission.landing], "what": ["launch", "landing"]})

lines = alt.Chart(true).mark_line(strokeWidth=2.5).encode(
    x=alt.X("day:Q", title="Mission day (0 = launch, 180 = landing)"),
    y=alt.Y("change_pct:Q", title="Knee extension, % change from own baseline"),
    color=alt.Color("crew:N", scale=CREW, title=None, legend=alt.Legend(orient="top")),
    tooltip=["crew", "day", alt.Tooltip("change_pct:Q", format="+.1f")])
dots = alt.Chart(obs).mark_point(filled=True, size=70, color=INK, shape="diamond").encode(
    x="days:Q", y="change_pct:Q", tooltip=["cohort", "days", "change_pct", "n"])
rules = alt.Chart(landing).mark_rule(color=GRAY, strokeDash=[4, 3]).encode(x="day:Q", tooltip=["what"])
(rules + lines + dots).properties(width="container", height=300,
    title=alt.Title("Strength follows each person's status",
                    subtitle="Diamonds: bed-rest cohorts (no exercise, and Campaign 11's exercise group at -9%)"))

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 2. What the instruments report

The same people seen through the instruments, on NASA's schedule: strength every 30 days in flight, two baseline sessions before launch, two after landing. Each dot is one test with measurement noise (about 8% for strength, from repeat tests in the data). The faint line is the truth.

In [4]:
readings = mission.timeline(seed=10)
df = gs.to_dataframe(readings)
print(f"{len(df)} readings:", df.groupby("instrument").size().to_dict())
df[df.measure == "knee_extension_60"][["crew_id", "day_label", "value", "unit", "true_value", "adherence", "events"]].head(12)

4844 readings: {'cycle': 28, 'dxa': 96, 'dynamometer': 240, 'wearable': 4480}


,crew_id,day_label,value,unit,true_value,adherence,events
9,Ana,L-60,117.5,ft-lb,117.1,1.0,
22,Ben,L-60,136.7,ft-lb,139.6,0.0,
35,Caro,L-60,215.7,ft-lb,205.5,1.0,
48,Dani,L-60,149.1,ft-lb,148.9,1.0,
55,Ana,L-30,122.7,ft-lb,117.1,1.0,
62,Ben,L-30,146.6,ft-lb,139.6,0.0,
69,Caro,L-30,205.9,ft-lb,205.5,1.0,
76,Dani,L-30,146.5,ft-lb,148.9,1.0,
643,Ana,FD15,119.7,ft-lb,113.3,1.0,
655,Ben,FD15,125.0,ft-lb,126.9,0.0,


In [5]:
k = df[df.measure == "knee_extension_60"].copy()
k = k.assign(day=k.mission_day, pct=100 * k.value / k.groupby("crew_id").true_value.transform("first"), kind="reading")
truth = true.assign(pct=lambda d: 100 + d.change_pct, kind="truth").rename(columns={"crew": "crew_id"})
both = pd.concat([truth[["crew_id", "day", "pct", "kind"]],
                  k[["crew_id", "day", "pct", "kind", "day_label", "value", "true_value", "adherence", "events"]]])
line = alt.Chart().transform_filter(alt.datum.kind == "truth").mark_line(strokeWidth=1.2, opacity=0.45).encode(
    x=alt.X("day:Q", title="Mission day"), y=alt.Y("pct:Q", title="Knee extension, % of baseline", scale=alt.Scale(zero=False)),
    color=alt.Color("crew_id:N", scale=CREW, legend=None))
pts = alt.Chart().transform_filter(alt.datum.kind == "reading").mark_circle(size=55, opacity=0.95, stroke=SURFACE, strokeWidth=1).encode(
    x="day:Q", y="pct:Q", color=alt.Color("crew_id:N", scale=CREW, legend=None),
    tooltip=["crew_id", "day_label", alt.Tooltip("value:Q", title="reading (ft-lb)"),
             alt.Tooltip("true_value:Q", format=".1f", title="truth (ft-lb)"), "adherence", "events"])
alt.layer(line, pts, data=both).properties(width=220, height=170).facet(
    facet=alt.Facet("crew_id:N", title=None, sort=["Ana", "Ben", "Caro", "Dani"]), columns=2
).properties(title=alt.Title("Dynamometer readings around the true curve", subtitle="Dots: readings. Line: truth."))

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 3. The spread between people, against the data

300 simulated people with no exercise and 300 with full exercise, compared with the bed-rest cohorts. The band holds the middle 80% of people.

In [6]:
pop = gs.Mission("iss_180")
rows = []
for adh, label in ((0.0, "No exercise"), (1.0, "Full exercise")):
    for i in range(300):
        c = pop.add_crew(adherence=adh, seed=1000 + i)
        rows += [{"group": label, "day": d, "change": c.change("knee_extension_60", d)} for d in range(0, 181, 5)]
popdf = pd.DataFrame(rows)
q = popdf.groupby(["group", "day"]).change.quantile([0.1, 0.5, 0.9]).unstack().reset_index()
q.columns = ["group", "day", "p10", "p50", "p90"]
GROUP2 = alt.Scale(domain=["No exercise", "Full exercise"], range=[ORANGE, AQUA])
area = alt.Chart(q).mark_area(opacity=0.18).encode(
    x=alt.X("day:Q", title="Days without gravity"), y=alt.Y("p10:Q", title="Knee extension, % change"), y2="p90:Q",
    color=alt.Color("group:N", scale=GROUP2, title=None, legend=alt.Legend(orient="top")))
med = alt.Chart(q).mark_line(strokeWidth=2.5).encode(x="day:Q", y="p50:Q", color=alt.Color("group:N", scale=GROUP2),
    tooltip=["group", "day", alt.Tooltip("p10:Q", format="+.1f"), alt.Tooltip("p50:Q", format="+.1f"), alt.Tooltip("p90:Q", format="+.1f")])
data_pts = alt.Chart(obs).mark_point(filled=True, size=80, color=INK, shape="diamond").encode(
    x="days:Q", y="change_pct:Q", tooltip=["cohort", "days", "change_pct", "n"])
cut = alt.Chart(pd.DataFrame({"day": [90]})).mark_rule(color=GRAY, strokeDash=[4, 3]).encode(x="day:Q")
(area + med + cut + data_pts).properties(width="container", height=300,
    title=alt.Title("Simulated population vs the bed-rest cohorts",
                    subtitle="Band: 10th-90th percentile of 300 people. Dashed line: the data stops at 90 days."))

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [7]:
check = q[q.day.isin([30, 70])].pivot(index="day", columns="group", values="p50").round(1)
check["data, no exercise"] = [-16.2, "-24.2 (Campaign 11), -22.4 (Campaign 3)"]
check["data, with exercise"] = ["", -8.8]
check

group,Full exercise,No exercise,"data, no exercise","data, with exercise"
day,,,,
30,-5.5,-15.2,-16.2,
70,-7.7,-21.3,"-24.2 (Campaign 11), -22.4 (Campaign 3)",-8.8


## 4. Bone: DXA before and after flight only

There is no DXA scanner on the ISS, so on NASA's schedule bone is scanned before launch and at 7, 180 and 365 days after landing. Here the scanner also has a fault: from day 300 to 400 it reads 1.1% low, the same fault the real Campaign 3 scanner had in 2006. The reading does not show it; `truth` does.

In [8]:
faulty = gs.default_instruments(seed=20, faults={"dxa": [gs.Fault("offset", start=300, end=400, pct=-1.1)]})
tl = mission.timeline(instruments={"dxa": faulty["dxa"]})
hip = gs.to_dataframe(tl).query("measure == 'hip_bmd'")
with pd.option_context("display.precision", 3):
    display(hip[["crew_id", "day_label", "value", "true_value", "faults", "adherence"]])

,crew_id,day_label,value,true_value,faults,adherence
1,Ana,L-60,0.893,0.903,,1.0
7,Ben,L-60,0.881,0.886,,0.0
13,Caro,L-60,0.930,0.915,,1.0
19,Dani,L-60,1.139,1.132,,1.0
25,Ana,R+7,0.871,0.864,,1.0
31,Ben,R+7,0.822,0.810,,0.0
37,Caro,R+7,0.809,0.820,,1.0
43,Dani,R+7,1.053,1.062,,1.0
49,Ana,R+180,0.883,0.878,offset -1.1%,1.0
55,Ben,R+180,0.827,0.838,offset -1.1%,0.0


In [9]:
hip_true = pd.DataFrame([{"crew_id": c.id, "day": d, "pct": 100 + c.change("hip_bmd", d)}
                         for c in mission.crew for d in np.arange(-60, 546, 3.0)])
hip["pct"] = 100 * hip.value / hip.groupby("crew_id").true_value.transform("first")
l = alt.Chart(hip_true).mark_line(strokeWidth=2).encode(
    x=alt.X("day:Q", title="Mission day"), y=alt.Y("pct:Q", title="Total hip density, % of baseline", scale=alt.Scale(zero=False)),
    color=alt.Color("crew_id:N", scale=CREW, title=None, legend=alt.Legend(orient="top")))
p = alt.Chart(hip).mark_circle(size=70, stroke=SURFACE, strokeWidth=1).encode(
    x="mission_day:Q", y="pct:Q", color=alt.Color("crew_id:N", scale=CREW),
    tooltip=["crew_id", "day_label", "value", alt.Tooltip("true_value:Q", format=".3f"), "faults"])
flight = alt.Chart(pd.DataFrame({"a": [0], "b": [180]})).mark_rect(opacity=0.07, color=BLUE).encode(x="a:Q", x2="b:Q")
(flight + l + p).properties(width="container", height=280,
    title=alt.Title("Bone falls through the whole flight and takes a year or more to come back",
                    subtitle="Shaded: in flight. Dots: DXA scans (two of them taken during the 1.1% scanner fault)"))

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 5. Aerobic capacity and the cycle test

VO2peak readings on the in-flight schedule, then a full graded test for Ben (no exercise) before launch and on flight day 165. After deconditioning the same workload takes more heart beats, which is what Campaign 3 subjects showed (a median 27.5 bpm more).

In [10]:
cyc = gs.CycleErgometer(seed=5)
stages = pd.concat([pd.DataFrame(cyc.session(ben, d)["stages"]).assign(when=w)
                    for d, w in ((-30, "Before launch (L-30)"), (165, "Flight day 166"))])
WHEN = alt.Scale(domain=["Before launch (L-30)", "Flight day 166"], range=[GRAY, BLUE])
alt.Chart(stages).mark_line(point=alt.OverlayMarkDef(size=45, filled=True), strokeWidth=2).encode(
    x=alt.X("watts:Q", title="Cycle workload (W)"), y=alt.Y("hr_bpm:Q", title="Heart rate (bpm)", scale=alt.Scale(zero=False)),
    color=alt.Color("when:N", scale=WHEN, title=None, legend=alt.Legend(orient="top")),
    tooltip=["when", "watts", "hr_bpm", "vo2_ml_kg_min", "rpe_borg"]
).properties(width="container", height=260,
             title=alt.Title("Ben's graded cycle test: shorter, with a higher heart rate at every stage",
                             subtitle="Stage values use the ACSM cycling equation and Tanaka's maximum heart rate"))

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [11]:
v = df[df.measure == "vo2peak"].pivot_table(index="day_label", columns="crew_id", values="value", sort=False)
v

crew_id,Ana,Ben,Caro,Dani
day_label,,,,
L-60,60.0,45.4,46.3,43.1
L-30,44.1,43.7,42.7,47.5
FD15,55.3,43.0,42.0,44.7
FD76,50.3,37.0,34.3,39.3
FD166,54.3,34.4,45.7,42.1
R+5,52.0,37.2,41.3,36.6
R+30,59.3,45.8,47.7,44.9


## 6. The wearable

Daily resting heart rate (fitted on Campaign 11's supine values) and one day of minute-by-minute heart rate. The daily shape (rhythm, sleep, two exercise blocks) is assumed; the resting level comes from the model.

In [12]:
rhr = df[df.measure == "resting_hr"]
alt.Chart(rhr).mark_line(strokeWidth=1.5).encode(
    x=alt.X("mission_day:Q", title="Mission day"), y=alt.Y("value:Q", title="Resting heart rate (bpm)", scale=alt.Scale(zero=False)),
    color=alt.Color("crew_id:N", scale=CREW, title=None, legend=alt.Legend(orient="top")),
    tooltip=["crew_id", "day_label", "value", "adherence"]
).properties(width="container", height=240, title=alt.Title("Daily resting heart rate", subtitle="Wearable, one reading a day"))

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [13]:
trace = pd.DataFrame(gs.Wearable(seed=2).session(caro, 40)["trace"]).assign(hour=lambda d: d.minute / 60)
alt.Chart(trace).mark_line(strokeWidth=1, color=BLUE).encode(
    x=alt.X("hour:Q", title="Hour of the day", scale=alt.Scale(domain=[0, 24])), y=alt.Y("hr_bpm:Q", title="Heart rate (bpm)"),
    tooltip=[alt.Tooltip("hour:Q", format=".2f"), "hr_bpm"]
).properties(width="container", height=200, title=alt.Title("Caro, flight day 41: 24 hours from the wearable"))

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 7. A Mars mission

The same crew on the Mars preset: 180 days of transit, 500 days on the surface at 0.38 g, 180 days back. The data has nothing on partial gravity or on missions this long, so every reading from the surface on is flagged `extrapolated`. Partial gravity here scales the unloading by 1 - g, an assumption you can replace.

In [14]:
mars = gs.Mission("mars")
for c in mission.crew:
    mars.add_crew(c.id, sex=c.sex, adherence=c.adherence_plan, responder=c.responder, seed=c.seed)
mdays = np.arange(-30, 1226, 5.0)
mt = pd.DataFrame([{"crew": c.id, "day": d, "change": c.change("knee_extension_60", d),
                    "extrapolated": c.extrapolated("knee_extension_60", d)} for c in mars.crew for d in mdays])
phases = pd.DataFrame([{"phase": p.name, "start": p.start, "end": p.end} for p in mars.phases if p.gravity_g < 1])
rect = alt.Chart(phases).mark_rect(opacity=0.08, color=BLUE).encode(x="start:Q", x2="end:Q", tooltip=["phase"])
line = alt.Chart(mt).mark_line(strokeWidth=2).encode(
    x=alt.X("day:Q", title="Mission day"), y=alt.Y("change:Q", title="Knee extension, % change"),
    color=alt.Color("crew:N", scale=CREW, title=None, legend=alt.Legend(orient="top")),
    strokeDash=alt.StrokeDash("extrapolated:N", legend=None, scale=alt.Scale(domain=[False, True], range=[[1, 0], [5, 3]])),
    tooltip=["crew", "day", alt.Tooltip("change:Q", format="+.1f"), "extrapolated"])
(rect + line).properties(width="container", height=280,
    title=alt.Title("Mars: partial recovery on the surface, loss again on the way home",
                    subtitle="Shaded: transit (0 g) and surface (0.38 g). Dashed: past what the data covers"))

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## 8. Into the 3D page

The readings go into a `DataService`, which runs every one through the ingestion pipeline (clean, check limits, convert units). The `HealthService` turns what the DataService holds into the JSON the 3D page reads (contract version 1). It uses only what the instruments reported, never the truth. Scores colour the body: 0 is the pre-flight reference, 100 is as much loss as bed-rest subjects had with no exercise at all. `python -m gravsim.serve` serves the same JSON at `/api/v1/dashboard`.

In [15]:
data = gs.DataService()
data.ingest(readings)
dash = gs.HealthService(data).dashboard(caro, day=110)
gs.validate_dashboard(dash)
pd.DataFrame(dash["summary"]["items"])[["title", "changePercent", "description"]]

,title,changePercent,description
0,Quadriceps,-25.4,Knee extension strength at 60°/s (dynamometer)...
1,Hamstrings,-7.2,"Knee flexion strength at 60°/s (dynamometer), ..."
2,Calves,-25.3,Ankle plantarflexion strength at 30°/s (dynamo...
3,Shins,-15.4,Ankle dorsiflexion strength at 30°/s (dynamome...
4,Back,-11.7,Trunk extension strength at 60°/s (dynamometer...
5,Arms,NaN,"Arm lean mass (DXA), latest L-60, compared wit..."


In [16]:
print(dash["summary"]["title"], "|", dash["summary"]["notice"])
print("scores:", dash["body"]["scores"])
print("insight:", dash["charts"][0]["insight"])

Caro · FD111 | Synthetic data from gravsim 0.1.0, built on NASA bed-rest studies. Not a health assessment.
scores: {'quadriceps_l': 100.0, 'quadriceps_r': 100.0, 'hamstrings_l': 52.9, 'hamstrings_r': 52.9, 'calf_l': 98.3, 'calf_r': 98.3, 'shin_muscles_l': 100.0, 'shin_muscles_r': 100.0, 'back_muscles_l': 66.1, 'back_muscles_r': 66.1}
insight: -25% since pre-flight. After 110 days without gravity, bed-rest subjects changed about -23% without exercise and -8% with it.
